# Eksperimen Model Rekomendasi Harga Jual Kemasan (Digipack.id)
### Riset Skripsi Informatika: Pendekatan Hibrida Machine Learning & Rule-Based Margin Guardrail

Notebook ini mendokumentasikan seluruh tahapan penelitian:
1. **Exploratory Data Analysis (EDA)** terhadap distribusi harga vendor (COGS), kuantitas pesanan, dan rasio markup.
2. **Feature Engineering** (kategori, material, ukuran, warna, print type, log qty, tier qty).
3. **Validasi Berbasis Waktu** (Time-series Split: Bulan 1-5 Train, Bulan 6 Test).
4. **Komparasi 4 Model**: Baseline Kategori, Linear Regression, Random Forest, dan LightGBM.
5. **Analisis Kuantil Residual** untuk pembentukan rentang rekomendasi harga (Min-Max Interval).
6. **Explainability dengan SHAP** untuk interpretasi faktor penentu harga.

In [3]:
# Install dependensi jika dijalankan di Google Colab atau environment baru
# %pip install lightgbm shap scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 1. Load Data Historis Penjualan (Anonim)

In [4]:
# Load data penjualan yang telah dianonimkan
df = pd.read_csv("../data/historical_sales_cleaned.csv")
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df["log_qty"] = np.log1p(df["qty"])
print(f"Total baris: {len(df)}")
df.head()

Total baris: 350


,so_code,order_date,customer_code,category,product_name,material,size,color,print_type,qty,vendor_code,vendor_cost,selling_price,markup_ratio,margin_pct,is_completed,sales_rep,month,log_qty,qty_tier
0,SO-202609-0001,2026-09-17,CLIENT_006,Paper Bag,White Satchel Paper Bag,Greaseproof 50,11 x 4 x 11,2/0,Sablon 1W2S,500,VENDOR_SP0001,632.78,1097.81,1.7349,0.7349,True,Salma Laila,9,6.216606,tier_small_<1k
1,SO-202605-0002,2026-05-24,CLIENT_014,Paper Bag,Brown Handle Paper Bag,Brown Kraft 80,20 x 9 x 21,0/0,Polos,500,VENDOR_SP0001,1085.60,1886.66,1.7379,0.7379,True,Shalsabilla Redwina,5,6.216606,tier_small_<1k
2,SO-202609-0003,2026-09-17,CLIENT_027,Paper Bag,Brown Handle Paper Bag,Brown Kraft 125,11 x 11 x 28,0/0,Polos,15000,VENDOR_SP0001,1297.80,1974.21,1.5212,0.5212,True,Shalsabilla Redwina,9,9.615872,tier_bulk_10k+
3,SO-202605-0004,2026-05-02,CLIENT_011,Paper Bag,Brown Handle Paper Bag,Brown Kraft 125,20 x 9 x 29,2/0,Sablon 1W2S,15000,VENDOR_SP0001,1622.61,2525.11,1.5562,0.5562,True,Salma Laila,5,9.615872,tier_bulk_10k+
4,SO-202605-0005,2026-05-24,CLIENT_025,Paper Bag,Brown Handle Paper Bag,Brown Kraft 80,20 x 9 x 21,2/0,Sablon 1W2S,4000,VENDOR_SP0001,1417.32,2416.39,1.7049,0.7049,True,Shalsabilla Redwina,5,8.294300,tier_medium_1k-5k


## 2. Eksplorasi Data & Statistik Deskriptif

In [5]:
print("Statistik Markup dan Margin:")
df[["qty", "vendor_cost", "selling_price", "markup_ratio", "margin_pct"]].describe()

Statistik Markup dan Margin:


,qty,vendor_cost,selling_price,markup_ratio,margin_pct
count,350.000000,350.000000,350.000000,350.000000,350.000000
mean,6380.000000,928.779686,1538.597029,1.653648,0.653648
std,6009.057156,640.015829,1072.539679,0.064230,0.064230
min,500.000000,107.000000,180.870000,1.490000,0.490000
25%,1500.000000,437.985000,727.205000,1.612700,0.612700
50%,4000.000000,718.630000,1192.590000,1.654000,0.654000
75%,10000.000000,1266.572500,2051.307500,1.700050,0.700050
max,20000.000000,3178.080000,5828.920000,1.881400,0.881400


## 3. Validasi Berbasis Waktu (Chronological Time-Series Split)

In [6]:
df_sorted = df.sort_values("order_date").reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.80)
train_df = df_sorted.iloc[:split_idx].copy()
test_df = df_sorted.iloc[split_idx:].copy()

features_cat = ["category", "material", "size", "color", "print_type", "qty_tier"]
features_num = ["qty", "log_qty", "vendor_cost", "month"]
all_features = features_cat + features_num

X_train = train_df[all_features]
y_train = train_df["markup_ratio"]
cost_train = train_df["vendor_cost"]

X_test = test_df[all_features]
y_test = test_df["markup_ratio"]
cost_test = test_df["vendor_cost"]
actual_price_test = test_df["selling_price"]
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

Train size: 280, Test size: 70


## 4. Pelatihan & Komparasi 4 Model

In [7]:
# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), features_cat),
        ("num", StandardScaler(), features_num)
    ]
)

# Baseline
cat_mean = train_df.groupby("category")["markup_ratio"].mean().to_dict()
global_mean = train_df["markup_ratio"].mean()
base_pred_markup = test_df["category"].map(cat_mean).fillna(global_mean)
base_pred_price = base_pred_markup * cost_test

models = {
    "Baseline (Rata-rata)": (None, base_pred_markup, base_pred_price),
    "Linear Regression": (LinearRegression(), None, None),
    "Random Forest": (RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42), None, None),
    "LightGBM": (lgb.LGBMRegressor(n_estimators=120, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1), None, None)
}

benchmark_results = []
for name, (reg, p_markup, p_price) in models.items():
    if reg is not None:
        pipe = Pipeline([("prep", preprocessor), ("reg", reg)])
        pipe.fit(X_train, y_train)
        p_markup = pipe.predict(X_test)
        p_price = p_markup * cost_test
    mae = mean_absolute_error(actual_price_test, p_price)
    rmse = np.sqrt(mean_squared_error(actual_price_test, p_price))
    mape = mean_absolute_percentage_error(actual_price_test, p_price) * 100
    below_margin = ((p_markup < 1.67).sum() / len(p_markup)) * 100
    benchmark_results.append({
        "Model": name,
        "MAE (Rp)": round(mae, 2),
        "RMSE (Rp)": round(rmse, 2),
        "MAPE (%)": round(mape, 2),
        "% Prediksi di Bawah Target 67%": round(below_margin, 2)
    })

res_df = pd.DataFrame(benchmark_results)
res_df

,Model,MAE (Rp),RMSE (Rp),MAPE (%),% Prediksi di Bawah Target 67%
0,Baseline (Rata-rata),44.38,62.97,3.08,100.00
1,Linear Regression,33.39,49.04,2.23,58.57
2,Random Forest,32.57,47.42,2.10,75.71
3,LightGBM,32.53,47.08,2.07,74.29


## 5. Analisis Rentang Harga Kuantil Residual

In [8]:
best_pipe = Pipeline([("prep", preprocessor), ("reg", models["LightGBM"][0])])
best_pipe.fit(X_train, y_train)
train_pred_markup = best_pipe.predict(X_train)
residuals = y_train.values - train_pred_markup
q10 = np.percentile(residuals, 10)
q90 = np.percentile(residuals, 90)
print(f"Residual Error Quantile 10%: {q10:.4f}")
print(f"Residual Error Quantile 90%: {q90:.4f}")

Residual Error Quantile 10%: -0.0401
Residual Error Quantile 90%: 0.0430


## 6. Interpretasi SHAP: Faktor Penentu Harga

In [9]:
explainer = shap.TreeExplainer(best_pipe.named_steps["reg"])
X_test_prep = best_pipe.named_steps["prep"].transform(X_test)
shap_values = explainer.shap_values(X_test_prep)
feature_names = list(best_pipe.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(features_cat)) + features_num
shap.summary_plot(shap_values, X_test_prep, feature_names=feature_names, plot_type="bar")

ImportError: matplotlib is not installed so plotting is not available! Run `pip install matplotlib` to fix this.